In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm
import time

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")

batch_size = 64         # Large batch = better GPU utilization
num_epochs = 10          # You can increase to 20 if stable
learning_rate = 1e-4
num_workers = 4  

🚀 Using device: cuda


In [3]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


In [4]:
train_data = datasets.ImageFolder(root="../dataset/archive/Dataset/Train", transform=train_transform)
val_data = datasets.ImageFolder(root="../dataset/archive/Dataset/Test", transform=val_transform)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

In [5]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# Freeze all layers first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last few blocks for fine-tuning
for name, param in model.features.named_parameters():
    if '5.' in name or '6.' in name:   # last blocks of the network
        param.requires_grad = True

# Replace classifier head
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, 2)   # 2 classes: real/fake
)

model = model.to(device)

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)

In [7]:
best_acc = 0.0
start_time = time.time()

for epoch in range(num_epochs):
    print(f"\n🌟 Epoch {epoch+1}/{num_epochs}")
    print("-" * 40)

    model.train()
    running_loss, correct, total = 0.0, 0, 0

    loop = tqdm(train_loader, desc="Training", leave=False)
    for images, labels in loop:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        running_loss += loss.item()

        loop.set_postfix(loss=loss.item(), acc=(100 * correct / total))

    # Validation phase
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total
    print(f"✅ Epoch [{epoch+1}/{num_epochs}] - Loss: {running_loss/len(train_loader):.4f}, Val Acc: {val_acc:.2f}%")

    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")

end_time = time.time()
print(f"\n🏁 Training completed in {(end_time - start_time)/60:.2f} minutes")
print(f"🏆 Best Validation Accuracy: {best_acc:.2f}%")

# ==========================================================
# 🔍 SAVE FINAL MODEL
# ==========================================================
torch.save(model.state_dict(), "final_model.pth")
print("✅ Model saved successfully as 'final_model.pth'")


🌟 Epoch 1/10
----------------------------------------


✅ Epoch [1/10] - Loss: 0.1337, Val Acc: 89.38%

🌟 Epoch 2/10
----------------------------------------


✅ Epoch [2/10] - Loss: 0.0677, Val Acc: 90.20%

🌟 Epoch 3/10
----------------------------------------


✅ Epoch [3/10] - Loss: 0.0551, Val Acc: 88.90%

🌟 Epoch 4/10
----------------------------------------


✅ Epoch [4/10] - Loss: 0.0472, Val Acc: 90.17%

🌟 Epoch 5/10
----------------------------------------


...

✅ Epoch [6/10] - Loss: 0.0391, Val Acc: 89.43%

🌟 Epoch 7/10
----------------------------------------


✅ Epoch [7/10] - Loss: 0.0356, Val Acc: 89.95%

🌟 Epoch 8/10
----------------------------------------


✅ Epoch [8/10] - Loss: 0.0330, Val Acc: 90.25%

🌟 Epoch 9/10
----------------------------------------


✅ Epoch [9/10] - Loss: 0.0314, Val Acc: 87.46%

🌟 Epoch 10/10
----------------------------------------


✅ Epoch [10/10] - Loss: 0.0303, Val Acc: 88.86%

🏁 Training completed in 106.63 minutes
🏆 Best Validation Accuracy: 90.54%
✅ Model saved successfully as 'final_model.pth'


In [20]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import numpy as np
from efficientnet_pytorch import EfficientNet

# ====================================
# 1️⃣ Setup Device (GPU if available)
# ====================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Using device: {device}")

# ====================================
# 2️⃣ Define the Deepfake Classifier
# ====================================
class DeepfakeClassifier(nn.Module):
    def __init__(self):
        super(DeepfakeClassifier, self).__init__()
        # Load EfficientNet-B0 pretrained model
        self.base_model = EfficientNet.from_pretrained('efficientnet-b0')
        in_features = self.base_model._fc.in_features
        self.base_model._fc = nn.Linear(in_features, 2)  # Binary classifier (Real/Fake)

    def forward(self, x):
        return self.base_model(x)

# ====================================
# 3️⃣ Load the Trained Model
# ====================================
model = DeepfakeClassifier().to(device)
model.load_state_dict(torch.load("final_model.pth", map_location=device), strict  = False)
model.eval()
print("✅ Model loaded successfully!")

# ====================================
# 4️⃣ Define Image Prediction Function
# ====================================
def predict_image(model, image_path, device):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    img = Image.open(image_path).convert('RGB')
    x = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    classes = [ 'Fake','Real']
    pred_class = classes[np.argmax(probs)]
    confidence = probs.max() * 100

    print(f"\n🧾 Image: {image_path}")
    print(f"🎯 Prediction: {pred_class}")
    print(f"💪 Confidence: {confidence:.2f}%")

# ====================================
# 5️⃣ Run Prediction on Example Image
# ====================================
if __name__ == "__main__":
    # Change this path to test your own image
    image_path = "../dataset/archive/Dataset/Test/Real/real_1.jpg"
    predict_image(model, image_path, device)


🔥 Using device: cuda
Loaded pretrained weights for efficientnet-b0
✅ Model loaded successfully!

🧾 Image: ../dataset/archive/Dataset/Test/Real/real_1.jpg
🎯 Prediction: Real
💪 Confidence: 52.32%


C:\Users\Drishi\AppData\Local\Temp\ipykernel_69268\2055840548.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("final_model.pth", map_lo

In [14]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import numpy as np
from efficientnet_pytorch import EfficientNet

# ====================================
# 1️⃣ Setup Device (GPU if available)
# ====================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Using device: {device}")

# ====================================
# 2️⃣ Define the Deepfake Classifier
# ====================================
class DeepfakeClassifier(nn.Module):
    def __init__(self):
        super(DeepfakeClassifier, self).__init__()
        # Load EfficientNet-B0 pretrained model
        self.base_model = EfficientNet.from_pretrained('efficientnet-b0')
        in_features = self.base_model._fc.in_features
        self.base_model._fc = nn.Linear(in_features, 2)  # Binary classifier (Real/Fake)

    def forward(self, x):
        return self.base_model(x)

# ====================================
# 3️⃣ Load the Trained Model
# ====================================
model = DeepfakeClassifier().to(device)
model.load_state_dict(torch.load("final_model.pth", map_location=device), strict  = False)
model.eval()
print("✅ Model loaded successfully!")

# ====================================
# 4️⃣ Define Image Prediction Function
# ====================================
def predict_image(model, image_path, device):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    img = Image.open(image_path).convert('RGB')
    x = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    classes = [ 'Fake','Real']
    pred_class = classes[np.argmax(probs)]
    confidence = probs.max() * 100

    print(f"\n🧾 Image: {image_path}")
    print(f"🎯 Prediction: {pred_class}")
    print(f"💪 Confidence: {confidence:.2f}%")

# ====================================
# 5️⃣ Run Prediction on Example Image
# ====================================
if __name__ == "__main__":
    # Change this path to test your own image
    image_path = "../dataset/archive/Dataset - Copy/ppic/ph7.png"
    predict_image(model, image_path, device)


🔥 Using device: cuda
Loaded pretrained weights for efficientnet-b0
✅ Model loaded successfully!

🧾 Image: ../dataset/archive/Dataset - Copy/ppic/ph7.png
🎯 Prediction: Fake
💪 Confidence: 55.93%


C:\Users\Drishi\AppData\Local\Temp\ipykernel_28744\2284277180.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("final_model.pth", map_lo

In [11]:
import cv2
import torch
from torchvision import transforms, models
import torch.nn as nn
from PIL import Image
import numpy as np
from tqdm import tqdm

# -----------------------------
# 🔧 Load Trained Model
# -----------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = models.efficientnet_b0(pretrained=False)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model.load_state_dict(torch.load('final_model.pth', map_location=device), strict  = False)
model = model.to(device)
model.eval()

# -----------------------------
# 🧠 Image Transformations
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# -----------------------------
# 🎞️ Function to Process Video
# -----------------------------
def detect_deepfake_in_video(video_path, frame_skip=5, display=False):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("❌ Error: Cannot open video.")
        return

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    predictions = []

    print(f"🎬 Processing {frame_count} frames at {fps} FPS...")

    for i in tqdm(range(frame_count)):
        ret, frame = cap.read()
        if not ret:
            break

        # Skip frames for faster processing
        if i % frame_skip != 0:
            continue

        # Convert to PIL and preprocess
        img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        img_tensor = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(img_tensor)
            pred = torch.argmax(output, dim=1).item()
            predictions.append(pred)

        # Optionally show live results
        if display:
            label = "Fake" if pred == 1 else "Real"
            color = (0, 0, 255) if pred == 1 else (0, 255, 0)
            cv2.putText(frame, f"Prediction: {label}", (20, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
            cv2.imshow("Deepfake Detector", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    if display:
        cv2.destroyAllWindows()

    # -----------------------------
    # 📊 Aggregate Results
    # -----------------------------
    real_count = predictions.count(1)
    fake_count = predictions.count(0)
    total = len(predictions)

    print("\n📊 Summary:")
    print(f"🟢 Real frames: {real_count}")
    print(f"🔴 Fake frames: {fake_count}")
    print(f"📽️ Total analyzed frames: {total}")

    # Decide final verdict
    if fake_count / total > 0.5:
        print("\n🚨 Final Verdict: FAKE VIDEO ⚠️")
    else:
        print("\n✅ Final Verdict: REAL VIDEO ✔️")

# -----------------------------
# ▶️ Run Example
# -----------------------------
if __name__ == "__main__":
    video_path = "../dataset/archive/Dataset - Copy/pvid/vid10.mp4"   # change this to your video file
    detect_deepfake_in_video(video_path, frame_skip=10, display=True)


C:\Users\Drishi\AppData\Local\Temp\ipykernel_28744\2379347836.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('final_model.pth', map_lo

🎬 Processing 3537 frames at 23 FPS...


100%|█████████▉| 3535/3537 [00:28<00:00, 124.77it/s]


📊 Summary:
🟢 Real frames: 195
🔴 Fake frames: 159
📽️ Total analyzed frames: 354

✅ Final Verdict: REAL VIDEO ✔️
